# Velora Hotel Melbourne — Housekeeping Operations Analytics

**Author:** Ashwini Harikumar

**Tools:** Python, pandas, matplotlib, seaborn

**Purpose:** Exploratory data analysis of housekeeping operational data to identify root causes of overtime, room readiness failures, and staffing imbalances.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded successfully')

In [ ]:
roster      = pd.read_csv('../data/staff_roster.csv', parse_dates=['shift_date'])
assignments = pd.read_csv('../data/room_assignments.csv', parse_dates=['shift_date'])
checkouts   = pd.read_csv('../data/checkout_log.csv', parse_dates=['checkout_date'])
status      = pd.read_csv('../data/room_status_log.csv', parse_dates=['log_date'])

print(f'staff_roster:     {roster.shape[0]} rows')
print(f'room_assignments: {assignments.shape[0]} rows')
print(f'checkout_log:     {checkouts.shape[0]} rows')
print(f'room_status_log:  {status.shape[0]} rows')

In [ ]:
for name, df in [('staff_roster', roster), ('room_assignments', assignments),
                 ('checkout_log', checkouts), ('room_status_log', status)]:
    nulls = df.isnull().sum().sum()
    print(f'{name}: {nulls} null values')

In [ ]:
ot_by_day = roster.groupby('day_of_week')['overtime_hours'].sum().reset_index()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
ot_by_day['day_of_week'] = pd.Categorical(ot_by_day['day_of_week'], categories=day_order, ordered=True)
ot_by_day = ot_by_day.sort_values('day_of_week')

fig, ax = plt.subplots()
bars = ax.bar(ot_by_day['day_of_week'], ot_by_day['overtime_hours'],
              color=['#e74c3c' if d in ['Monday','Friday'] else '#3498db' for d in ot_by_day['day_of_week']])
ax.set_title('Total Overtime Hours by Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Total Overtime Hours')
ax.bar_label(bars, fmt='%.1f', padding=3)
plt.tight_layout()
plt.show()
print('Insight: Monday and Friday account for the majority of overtime hours.')

In [ ]:
ot_by_staff = roster.groupby('staff_name').agg(
    total_overtime=('overtime_hours','sum'),
    total_rooms=('rooms_assigned_count','sum')
).reset_index().sort_values('total_overtime', ascending=False)

fig, ax = plt.subplots()
bars = ax.barh(ot_by_staff['staff_name'], ot_by_staff['total_overtime'], color='#e74c3c')
ax.set_title('Total Overtime Hours by Staff Member', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Overtime Hours')
ax.bar_label(bars, fmt='%.1f', padding=3)
plt.tight_layout()
plt.show()
print(ot_by_staff.to_string(index=False))

In [ ]:
floor_load = roster.groupby('floor_assigned').agg(
    avg_rooms=('rooms_assigned_count','mean'),
    avg_overtime=('overtime_hours','mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(floor_load['floor_assigned'].astype(str), floor_load['avg_rooms'], color='#2980b9')
axes[0].axhline(floor_load['avg_rooms'].mean(), color='red', linestyle='--', label='Average')
axes[0].set_title('Avg Rooms Assigned Per Shift by Floor', fontweight='bold')
axes[0].set_xlabel('Floor')
axes[0].set_ylabel('Avg Rooms')
axes[0].legend()

axes[1].bar(floor_load['floor_assigned'].astype(str), floor_load['avg_overtime'],
            color=['#e74c3c' if x > floor_load['avg_overtime'].mean() else '#27ae60' for x in floor_load['avg_overtime']])
axes[1].axhline(floor_load['avg_overtime'].mean(), color='black', linestyle='--', label='Average')
axes[1].set_title('Avg Overtime Hours Per Shift by Floor', fontweight='bold')
axes[1].set_xlabel('Floor')
axes[1].set_ylabel('Avg Overtime Hours')
axes[1].legend()

plt.tight_layout()
plt.show()
print('Insight: Floors 3, 5 and 7 are overloaded. Floors 1, 2 and 6 are under-utilised.')

In [ ]:
bucket_stats = checkouts.groupby('checkout_hour_bucket').agg(
    total_checkouts=('checkout_id','count'),
    avg_clearance_min=('minutes_to_clear','mean')
).reset_index()

bucket_order = ['09:00-10:00','10:00-11:00','11:00-12:00']
bucket_stats['checkout_hour_bucket'] = pd.Categorical(bucket_stats['checkout_hour_bucket'], categories=bucket_order, ordered=True)
bucket_stats = bucket_stats.sort_values('checkout_hour_bucket')

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.bar(bucket_stats['checkout_hour_bucket'], bucket_stats['total_checkouts'], color='#3498db', alpha=0.7, label='Checkout Volume')
ax2.plot(bucket_stats['checkout_hour_bucket'], bucket_stats['avg_clearance_min'], color='#e74c3c', marker='o', linewidth=2, label='Avg Clearance (min)')
ax1.set_title('Checkout Volume vs Average Clearance Time by Hour', fontsize=13, fontweight='bold')
ax1.set_xlabel('Hour Bucket')
ax1.set_ylabel('Number of Checkouts', color='#3498db')
ax2.set_ylabel('Avg Clearance Time (min)', color='#e74c3c')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()
print('Insight: The 10-11am window has the highest checkout volume AND slowest clearance times.')

In [ ]:
sla_pct = round(status['sla_met'].eq('Yes').mean() * 100, 1)
print(f'Overall SLA Compliance Rate: {sla_pct}%')

sla_by_floor = status.groupby('floor').apply(
    lambda x: round(x['sla_met'].eq('Yes').mean() * 100, 1)
).reset_index()
sla_by_floor.columns = ['floor', 'sla_compliance_pct']

fig, ax = plt.subplots()
colors = ['#e74c3c' if v < 70 else '#f39c12' if v < 85 else '#27ae60' for v in sla_by_floor['sla_compliance_pct']]
bars = ax.bar(sla_by_floor['floor'].astype(str), sla_by_floor['sla_compliance_pct'], color=colors)
ax.axhline(85, color='green', linestyle='--', linewidth=1.5, label='Target 85%')
ax.axhline(sla_pct, color='navy', linestyle=':', linewidth=1.5, label=f'Overall {sla_pct}%')
ax.set_title('Room Readiness SLA Compliance by Floor', fontsize=13, fontweight='bold')
ax.set_xlabel('Floor')
ax.set_ylabel('SLA Compliance (%)')
ax.set_ylim(0, 110)
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.legend()
plt.tight_layout()
plt.show()
print('Insight: Floors 5 and 7 are the worst SLA performers.')

In [ ]:
productivity = assignments.groupby('staff_name').agg(
    avg_actual=('minutes_taken','mean'),
    avg_standard=('standard_minutes','mean'),
    rooms_cleaned=('assignment_id','count')
).reset_index()
productivity['overrun'] = productivity['avg_actual'] - productivity['avg_standard']
productivity = productivity.sort_values('overrun')

fig, ax = plt.subplots()
x = range(len(productivity))
ax.bar(x, productivity['avg_actual'], label='Actual Avg (min)', color='#e74c3c', alpha=0.8)
ax.bar(x, productivity['avg_standard'], label='Standard (min)', color='#3498db', alpha=0.6)
ax.set_xticks(list(x))
ax.set_xticklabels(productivity['staff_name'], rotation=15)
ax.set_title('Avg Minutes Per Room: Actual vs Standard by Staff', fontsize=13, fontweight='bold')
ax.set_ylabel('Minutes Per Room')
ax.legend()
plt.tight_layout()
plt.show()
print(productivity[['staff_name','avg_actual','avg_standard','overrun','rooms_cleaned']].to_string(index=False))

In [ ]:
mon_fri_ot = roster[roster['day_of_week'].isin(['Monday','Friday'])]['overtime_hours'].sum()
total_ot = roster['overtime_hours'].sum()
mon_fri_pct = round(mon_fri_ot / total_ot * 100, 1)

peak_pct = round(checkouts[checkouts['checkout_hour_bucket'] == '10:00-11:00'].shape[0] / checkouts.shape[0] * 100, 1)

print('=' * 55)
print('   VELORA HOTEL - KEY FINDINGS SUMMARY')
print('=' * 55)
print(f'Overtime on Mon/Fri: {mon_fri_pct}% of all OT hours')
print(f'10-11am checkout share: {peak_pct}%')
print(f'Overall SLA compliance: {sla_pct}% (target: 85%)')
print('Floors 7 and 5: highest overtime and lowest SLA')
print('Floors 1 and 2: under-allocated relative to daily average')
print('=' * 55)
print()
print('RECOMMENDATIONS:')
print('1. Add 2 staff to Mon/Fri morning shifts')
print('2. Rebalance floor allocation - reduce load on F7 and F5')
print('3. Deploy a 3-person 10-11am checkout blitz team')
print('4. Pair bottom-quartile staff with top performers for coaching')